This assignment requires the installation of pandapower and associated packages with the line of code below (uncomment the first time you run the code if needed).

In [ ]:
# %pip install pandapower[all]
# %pip install plotly # Optional, for interactive plots

# Introduction

In this assignment, you will explore State Estimation (SE) by implementing the weighted-least-squares (WLS) SE algorithm and by testing it on the Svedala 32-bus transmission grid that you used in Aristo. It is strongly recommended that you read Sec. 3.1 of PSCDO provided in the course literature as this chapter covers everything you need for the implementation and construction of demonstration cases.

The assignment is divided in two parts:
<ol>
<li> Implementation of the WLS State Estimation Algorithm: You have to fill in some code to build the state estimator, and answer 6 questions. </li>
<li> Construction and Analysis of a Demonstration Case with faulty data: Follow the instructions to create (code) a demonstration case. Then, using the course literature, use a method to identify bad data and discuss the implications. </li>
</ol>

# Submission and Grading
Your submission for this assignment consists of this Notebook completed. You will be graded based on both your answers to the questions that you will find throughout this file, and the completion of the code. There are 7 questions, and the lines of code that you have to complete are indicated by #Fill this out/in.

The minimum requirement to pass this assignment is to have the state estimator running correctly.

<a id="0"></a> <br>
 # Table of Contents  
1. [**Implementing the WLS State Estimation Algorithm**](#1)  

    1.1 [Network Import and Admittance Matrix (code)](#2)     
    1.2 [Measurements and Weight Matrix (3 Questions + Code)](#3)   
    1.3 [WLS Estimator (Code)](#4)     
    1.4 [WLS Solver and Results (3 Questions + Code)](#5)    

2. [**Quality and Completeness of the Measurement Set** (code + questions)](#6) 



<a id="1"></a>
# 1. Implementing the Weighted Least Squares State Estimation algorithm
In this section, you will implement the WLS state estimation and run a simple test on the Sevdala power grid. To begin with, we need to import the functions and packages that will be used, load the test grid, and run a powerflow computation.

<a id="2"></a> 
## 1.1 - Network import & Admittance Matrix

### Svedala Network Model
**Fill in the code** to import the Svedala network from the CIM files available in the Folder Svedala_CIM, and optionally run a powerflow computation. After the import, we set a slack generator and import the geographical coordinates which were absent from the CIM files (GL file not defined).

In [ ]:
# Library Imports
import numpy as np; import copy; import json 
import pandapower as pp
from pandapower.plotting.plotly import vlevel_plotly, pf_res_plotly

# Custom Imports
from functions_base import get_PQ_inj, get_PQ_flow, get_dS_dV, get_dS_dV_flow, get_measurements_pu

# A lot of Future Warnings appear during the CIM import. We hide future warnings for readability. 
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Fill in to load the network from CIM files










# Set the Slack Generator and get the slack bus index
net.gen.loc[net.gen.name == "HALLAN_G1", "slack"] = True
slack_bus = int(net.gen.loc[net.gen.slack, 'bus'].iloc[0])
print(f"Slack Bus IDs: {slack_bus}")
# Load geo data
with open('geo_data.json', 'r') as f: coords = json.load(f)
net.bus['geo'] = [json.dumps({"type": "Point", "coordinates": coords[n]}) if n in coords else None for n in net.bus.name]

# vlevel_plotly(net, line_width=0.7);
pf_res_plotly(net, line_width=0.7);

### Admittance Matrix
The admittance matrix $Y_{bus}$ relates the voltages to the currents in the network through $I_{bus} = Y_{bus}\cdot V$, which gives $S = P + jQ = diag(V) \overline I_{bus}$.

Instead of manually building it, we let pandapower computing it when running the powerflow, then we retrieve it. Similarly, we get $Y_{f}$ and $Y_{t}$ that define the admittance matrices for the network branches (lines and transformers) in both directions (from and to). The admittance matrices are obtained using the function *make_y(net)* below.
Additionaly, *make_V(theta, Vm)* returns the complex voltage vector from the voltage magnitude and angle.

In [ ]:
def make_y(net):
    """  
    Ybus: Admittance matrix 
    Yf, Yt: branch admittance matrix
    """
    # Run power flow to get internal Ybus
    pp.runpp(net, enforce_q_lims=True) 
    ppc = net._ppc
    Ybus = ppc['internal']['Ybus'].toarray()
    Yf = ppc['internal']['Yf'].toarray()  # From/HV admittance matrix
    Yt = ppc['internal']['Yt'].toarray()  # To/LV admittance matrix      

    # Initialize shunt admittance matrix
    Ysh = np.zeros(Ybus.shape, dtype=complex)    
    # System parameters
    Sbase = net.sn_mva
    
    # Add shunt compononents to Ybus
    for i in range(len(net.shunt)):
        if not net.shunt.loc[i, 'in_service']:
            continue
        id_bus = net.shunt.loc[i, "bus"]   
        vn_shunt = net.shunt.loc[i, "vn_kv"]   # Vnom shunt
        step = net.shunt.loc[i, "step"]        # Step for variable shunts     
        
        # Shunt power: S = P + jQ      
        S_shunt = (net.shunt.loc[i, "p_mw"] + net.shunt.loc[i, "q_mvar"]*1j)*step        
        # Convert to admittance: Y = conj(S/V²)
        vref = net.bus.loc[id_bus, "vn_kv"]
        Ybase = Sbase/(vref**2)
        Ysh_shunt = np.conj(S_shunt/(vn_shunt**2))/Ybase    
        Ysh[id_bus, id_bus] -= Ysh_shunt 
    Ybus+= Ysh  # Add shunt admittances to Ybus
    return  Ybus, Yf, Yt

def make_V(theta, Vm):
    """ Return complex voltage vector """
    V = Vm*np.exp(1j*theta)
    return V

<a id="3"></a>
## 1.2 - Measurements and Weight Matrix

### Create Measurements
A state estimation is based on measurements. Hence, measurements are synthesised by adding noise to pandapower load flow results. 
The measurements generated by the function *create_measurement_vectors* and stored in the measurement vector $z$ include:
- Active and reactive power flows at both ends of transmission lines and transformers - ($\sigma_P$ = 50MW, $\sigma_Q$ = 10 MVar)
- Active and reactive power injections at buses - ($\sigma_P$ = 50MW, $\sigma_Q$ = 10 MVar)
- Voltage magnitudes at buses - ($\sigma_V$ = 0.004 p.u)

The function *get_measurements_pu(net)* defined in functions_base.py returns the measurement vector $z$ in p.u. and the vector of standard deviations $stds$. Note that the 'p' and 'q' measurements stored in the pandapower network net, are in MW/MVar.

The state vector of the Svedala power grid is given by:

\begin{align*}
    x^T=\begin{pmatrix}\theta_1 & \theta_2 & \cdots & \theta_5 & \theta_7 & \cdots & \theta_{32} & V_{m_1} & V_{m_2} & \cdots & V_{m_{32}} \end{pmatrix}
\end{align*}

where $\theta_i$ and $V_{m_i} are respectively the voltage angle and magnitude at bus $i$.

To relate the measurements to the state vector, we assume that the measurement vector $z$ is related to the states through some function $h(x)$. Since the measurements are not perfect, we further assume that the errors can be modelled by some term $e$, giving the expression $z = h(x) + e$.

In [ ]:
def create_measurement_vectors(net, v_stddev=0.004, p_stddev=50, q_stddev=10,
                                 add_noise=False, noise_zones=None,zero_injection_buses=[14,22,27,28]):
    
    """ Create measurement vectors for voltage magnitudes, active and reactive power injections, 
    and active and reactive power flows.
    !! It reinitializes the net.measurement DataFrame 
    """

    np.random.seed(5) # Ensure replicability (i.e. ensure that the same set of measurements is generated for the same stds)
    if noise_zones is not None:
        if isinstance(noise_zones, int):
            noise_zones = [noise_zones]
        elif not isinstance(noise_zones, list):
            noise_zones = None
    
    net.measurement = net.measurement[0:0] # Clear existing measurements
    pp.runpp(net, enforce_q_lims=True) # Run the loadflow
    
    # BUSES
 
    for busIndex in net.bus.index:     
        bus_zone = net.bus.loc[busIndex, 'zone']
        apply_noise = add_noise and (noise_zones is None or bus_zone in noise_zones)
            
        # Voltage measurement
        real_val = net.res_bus.vm_pu[busIndex]
        meas_val = np.random.normal(real_val, v_stddev) if apply_noise else real_val
        pp.create_measurement(net, "v", "bus", meas_val, v_stddev, element=busIndex)
            
        # Power injections
        real_val_p = net.res_bus.p_mw[busIndex]
        real_val_q = net.res_bus.q_mvar[busIndex]    
            
        if not (busIndex in zero_injection_buses):  # Do not create measurements for zero injection buses            
            if real_val_p != 0:
                meas_val = np.random.normal(real_val_p, p_stddev) if apply_noise else real_val_p
                pp.create_measurement(net, "p", "bus", meas_val, p_stddev, element=busIndex)                
            if real_val_q != 0:
                meas_val = np.random.normal(real_val_q, q_stddev) if apply_noise else real_val_q
                pp.create_measurement(net, "q", "bus", meas_val, q_stddev, element=busIndex)
    
    # LINES           
    for lineIndex in net.line.index:    
        if net.line.loc[lineIndex, 'in_service']:        
            lineIndex = int(lineIndex)
            from_bus = net.line.loc[lineIndex, 'from_bus']
            to_bus = net.line.loc[lineIndex, 'to_bus']
            from_zone = net.bus.loc[from_bus, 'zone']
            to_zone = net.bus.loc[to_bus, 'zone']
            apply_noise = add_noise and (noise_zones is None or from_zone in noise_zones or to_zone in noise_zones)
                
            for side in ['from', 'to']:                             
                real_val = net.res_line.loc[lineIndex, f'p_{side}_mw']
                meas_val = np.random.normal(real_val, p_stddev) if apply_noise else real_val
                pp.create_measurement(net, "p", "line", meas_val, p_stddev, element=lineIndex, side=side)
                        
                real_val = net.res_line.loc[lineIndex, f'q_{side}_mvar']
                meas_val = np.random.normal(real_val, q_stddev) if apply_noise else real_val
                pp.create_measurement(net, "q", "line", meas_val, q_stddev, element=lineIndex, side=side)
    
    # TRANSFORMERS
    for trafoIndex in net.trafo.index:
        if net.trafo.loc[trafoIndex, 'in_service']:
            trafoIndex = int(trafoIndex)
            hv_bus = net.trafo.loc[trafoIndex, 'hv_bus']
            lv_bus = net.trafo.loc[trafoIndex, 'lv_bus']
            hv_zone = net.bus.loc[hv_bus, 'zone']
            lv_zone = net.bus.loc[lv_bus, 'zone']
            apply_noise = add_noise and (noise_zones is None or hv_zone in noise_zones or lv_zone in noise_zones)
                
            for side in ['hv', 'lv']:
                real_val = net.res_trafo.loc[trafoIndex, f'p_{side}_mw']
                meas_val = np.random.normal(real_val, p_stddev) if apply_noise else real_val
                pp.create_measurement(net, "p", "trafo", meas_val, p_stddev, element=trafoIndex, side=side)
                    
                real_val = net.res_trafo.loc[trafoIndex, f'q_{side}_mvar']
                meas_val = np.random.normal(real_val, q_stddev) if apply_noise else real_val
                pp.create_measurement(net, "q", "trafo", meas_val, q_stddev, element=trafoIndex, side=side)       

### Questions + Code
1. Why is the voltage angle $\theta_6$ at the slack bus not part of the state vector? What's the purpose of it?

2. What assumptions do we have to make about the distribution of the errors $e$?

3. In the weighted least-squares estimator, the measurements are weighted by their inverse standard deviations through the inverse covariance matrix $R^{-1} = W$. Why?

**Complete the code** below to return the inverse covariance matrix with the function inverse_cov_matrix() below.

In [ ]:
def weight_matrix(stds): 
    """ Create the weight matrix W based on the standard deviations of the measurements """
    # Fill this in  
    W = np.diag(1.0 / np.square(stds))
    return W

<a id="4"></a>
## 1.3 - WLS Estimator
The WLS estimator aims to minimise the objective function J:
\begin{align*}
    J(x) &= \sum_{i=1}^m \frac{(z_i - h_i(x))^2}{R_i} = [z - h(x)]^T W[z-h(x)],
\end{align*}
where $h(x)$ is given by the set of non-linear equations that relate the states $x$ to the obtained measurements $z$.

In other words, it tries to minimise the sum of squared errors between the measurements and the power system model $h(x)$.

### Power System Model $h(x)$ - Code
 **Complete the code below** to get the matrix $h(x)$ from the function compute_h() below. 

In [ ]:
def compute_h(net, Ybus, Yf, Yt, V):    
    """  Compute the measurement function h(x) for all measurements """  
 
    S = get_PQ_inj(Ybus, V)  # Complex power injection at each bus  
    S_from, S_to = get_PQ_flow(net, Yf, Yt, V)  # Complex power flow on lines and transformers

    # Initialize h
    h = np.zeros(net.measurement.shape[0], dtype = float)   

    # Iterate through all measurements
    for index, measurement in net.measurement.iterrows():     

        meas_type = measurement['measurement_type']  # 'v', 'p', or 'q'
        element_type = measurement['element_type']   # 'bus', 'line', or 'trafo'
        element_id = int(measurement['element'])     # ID of the element
        element_side = measurement['side']           # 'from'/'to' for lines, 'hv'/'lv' for trafos       

        # Select the complex power for 'p' and 'q' measurements
        if element_type == 'trafo':
            element_id += len(net.line)
        if element_side in ['from', 'hv']:
            s_temp = S_from[element_id]
        elif element_side in ['to', 'lv']:
            s_temp = S_to[element_id]
        elif element_type == 'bus':
            s_temp = S[element_id]  

        #Fill this out  

            
    return h

### First order optimality condition

In order to minimise $J(x)$, we note that the first order optimality condition
\begin{align*}
    g(x) &= J'(x) =  \frac{\partial J(x)}{\partial x} = -H^T(x)W[z-h(x)] = 0,
\end{align*}
where $H(x) = \frac{\partial h(x)}{\partial x}$. $H(x)$ is a matrix of dimensions ($n\times m$), where $n$ is the number of measurements and $m$ is the number of state variables

### $H(x)$ - Code

**Fill the code** below to compute $H$ with the function *compute_H*. To help with this, three additional functions are created:
- *get_dh_dx_meas*: builds the rows of $H$ by computing $\frac{\partial M_i}{\partial \theta}$ and $\frac{\partial M_i}{\partial V_m}$, for **all** voltage angles and magnitudes, where $M_i$ is one measurement.
- *get_dS_dV*<sup>1</sup>: returns dS_dVm and dS_dtheta, the partial derivatives of the complex power injections $S$.
- *get_dS_dV_flow*<sup>1</sup>: returns dSf_dVm, dSf_dtheta, dSt_dVm, dSt_dtheta, the partial derivatives of the complex power flows in both directions. 

As a reminder:
\begin{align*}
    x^T=\begin{pmatrix}\theta_1 & \theta_2 & \cdots & \theta_5 & \theta_7 & \cdots & \theta_{32} & V_{m_1} & V_{m_2} & \cdots & V_{m_{32}} \end{pmatrix}
\end{align*}

<small>1. [TN2]  R. D. Zimmerman, "AC Power Flows, Generalized OPF Costs and
       their Derivatives using Complex Matrix Notation", MATPOWER
       Technical Note 2, February 2010.</small>

In [ ]:
def compute_H(net, Ybus, Yf, Yt, V, Vm, slack_bus=6):
    """ Compute the Jacobian matrix H(x) = dh/dx """
    # Initialization
    dh_dtheta = np.zeros((len(net.measurement), len(net.bus)))
    dh_dVm = np.zeros((len(net.measurement), len(net.bus)))

    # Get the partial derivatives of the complex power injections and flows
    dS_dVm, dS_dtheta = get_dS_dV(V, Vm, Ybus)
    dSf_dVm, dSf_dtheta, dSt_dVm, dSt_dtheta = get_dS_dV_flow(Yf, Yt, V, Vm, net)

    ### Fill in the code below to get dh_dtheta and dh_dVm and build H (do I specify that SLACK should be removed?) ###    

    return H

def get_dh_dx_meas(net, measurement, dS_dtheta, dS_dVm, dSf_dtheta, dSf_dVm, dSt_dtheta, dSt_dVm):
    """ Get the derivative of a single measurement
     dh_dtheta_temp: Row vector of partial derivatives of the measurement w.r.t. all voltage angles
     dh_dVm_temp: Row vector of partial derivatives of the measurement w.r.t. voltage magnitudes """
    
    # Extract measurement details
    meas_type = measurement['measurement_type']  # 'v', 'p', or 'q'
    element_type = measurement['element_type']   # 'bus', 'line', or 'trafo'
    element_id = int(measurement['element'])     # ID of the element
    element_side = measurement['side']           # 'from'/'to' for lines, 'hv'/'lv' for trafos

    # Initialize the row vector for this measurement   
    dh_dVm_temp = np.zeros(len(net.bus))
    dh_dtheta_temp = np.zeros(len(net.bus))

    if meas_type == 'v':
        dh_dVm_temp[element_id] = 1  # dVi/dVi  
        return dh_dtheta_temp, dh_dVm_temp
    
    if element_type == 'bus':
        dS_dtheta_temp = dS_dtheta[element_id,:]  # dS/dtheta
        dS_dVm_temp = dS_dVm[element_id,:]        # dS/dV
    else: # Line / trafo
        if element_type == 'trafo':
            element_id += len(net.line)
        if element_side in ['from', 'hv']:
            dS_dtheta_temp = dSf_dtheta[element_id,:]  # dS_from/dtheta
            dS_dVm_temp = dSf_dVm[element_id,:]     # dS_from/dV
        else: # to / lv
            dS_dtheta_temp = dSt_dtheta[element_id,:]  # dS_to/dtheta
            dS_dVm_temp = dSt_dVm[element_id,:]     # dS_to/dV

    if meas_type == 'p':
        dh_dtheta_temp = np.real(dS_dtheta_temp)  # dP/dtheta
        dh_dVm_temp = np.real(dS_dVm_temp)     # dP/dV
    else: # "q"
        dh_dtheta_temp = np.imag(dS_dtheta_temp)  # dQ/dtheta
        dh_dVm_temp = np.imag(dS_dVm_temp)     # dQ/dV  
    return dh_dtheta_temp, dh_dVm_temp

### Jacobian of the objective function, $g(x)$ - Code
Using the functions defined so far, we now have everything we need to compute the Jacobian $g(x)$ of the cost function, which was defined above. This is done in the function *compute_g()* below, which you will also need to **fill**.

In [ ]:
def compute_g(z, Rinv, H, h):
    # Fill this out to compute g
 
    return g

### Gain function ($G(x)$) - Code
We now have everything we need to run the state estimation algorithm. The function $g(x)$ can be expanded in a Taylor series around $x^k$ as
\begin{align*}
    g(x) &= g(x^k) + G(x^k)(x-x^k) + \mathrm{HOT},
\end{align*}
where we have defined the gain matrix $G(x^k)$ as
\begin{align*}
    G(x^k) &= \frac{\partial g(x^k)}{\partial x} = H^T(x^k)R^{-1}H(x^k).
\end{align*}
$G(x^k)$ is calculated in compute_G(), you **need to fill its expression below**.

In [ ]:
def compute_G(H, Rinv):
    # Fill this out to compute G

    return G    

<a id="5"></a>
## 1.4 - WLS Solver and Results

If the higher order terms (HOT) are neglected, the state vector can be calculated iteratively through
\begin{align*}
    x^{k+1} &= x^k - G(x^k)^{-1}g(x^k) ⇒ \Delta x^{k+1} = - G(x^k)^{-1}g(x^k)
\end{align*}

### WLS Algorithm - Code

Now that the necessary functions are implemented, we are ready to implement the state estimation algorithm. To do this, fill out the loop of the function *run_se()* below. **Fill in the code below** to recompute all matrices and the new estimation of the state vector after each iteration in the while loop.

In [ ]:
def run_se(net, z, W, Ybus, Yf, Yt, eps=1e-6, max_iter=10, slack_bus=6):
    """ Run state estimation """
    nbus = len(net.bus)
    
    # 1: Initialization state vector (flat start: angles = 0, voltages = 1.0 p.u.)  
    x = np.concatenate((np.zeros(nbus - 1), np.ones(nbus)))
    # Initialize iteration variables
    delta_x = 1e10 * np.ones(len(x))  
    k = 0  # Iteration counter 
    observable = True  # Flag to track observability    
    
    # 2: Iterate to converge (Try, except, if observable may not help to determine what cause the non-convergence (e.g. wrong code above))
    while    
    
        
        try: # Solve the system
       
        except Exception as e:         
            observable = False
            print(f"\n Solver failure at iteration {k}:", e)    
            print("The system may be unobservable or numerically ill-conditioned.\n")      
            break   

    # Print convergence information
    if observable:
        if np.max(np.abs(delta_x)) <= eps:
            print(f"\n CONVERGED in {k} iterations")
            print(f"Final max |delta x| = {np.max(np.abs(delta_x)):.2e}")          
        else:
            print(f"\n Maximum iterations ({max_iter}) reached without convergence")
            print(f"Current max |delta x| = {np.max(np.abs(delta_x)):.2e} (tolerance = {eps})") 
    return x

### Checking Results - Questions + Code
Once you have completed the algorithm, fill in the code cell below, run the SE algorithm, check your results, compare them with the theoretical results from the power flow calculation (State vector obtained from $x_{est}$ and Power injection using *get_PQ_inj(Ybus, V)*, where V is the Vector of complex Voltage).

Once you have compared the results, answer the following:

4. In real-world applications, how can the accuracy of the state estimation be improved and what mitigates the impact of local higher measurement errors?

5. In theory, how many measurements are necessary to perform the state estimation? Why more are often required?

6. What property of the measurement system guarantees that the network is observable and that a unique state estimate can be obtained?

In [ ]:
# Fill in the code below
slack_bus = int(net.gen.loc[net.gen.slack, 'bus'].iloc[0])


<a id="6"></a>
# 2. Quality of the measurement set

In this second part, you will use the state estimator from part 1 to demonstrate the effect of errors on the estimation accuracy, and how to detect bad data.

### Instructions
You are asked to do and answer the following:

<ol type="i">
 <li>Create a demonstration case in which you:</li>
    <ul>
        <li> Simulate a topological error so that there is a mismatch between the measurements and the grid model in the state estimator.</li>
        <li> Decrease the accuracy of a few localized measurements by increasing their standard deviation, and modifying their values accordingly.</li>
    </ul>
 <li>Compute the residuals and compare them with the measurement error.</li>
 <li>Using the residuals, identify which measurements are the least consistent with the estimated measurement accuracies.</li>
 <li>Delete the corresponding measurements and run the state estimator with corrected data.</li>
</ol>

### Question (Discussion/Analysis)

7. Describe what you notice throughout the demonstration case, and discuss the implications on the identification of bad data, and on the accuracy of state estimators: 
- How do the residuals compare with the measurement error? 
- What are the purpose and principle of the bad data detection method that you implemented?
- What are the limitations of the simple bad data detection & deletion process that was implemented?


### Additional information

You are free to create any useful complement/alternative function you want, but a few helper functions are provided to assist you:

Compute estimated values en error (in the code below):
<ul>
<li> <i>get_estimation(x, Ybus, Yf, Yt, net, slack_bus = 6)</i> (below): returns the voltage angles, magnitudes and complex values, as well as the active and reactive power flows computed from the state vector x.</li>
<li> <i>get_error(X_est, X_true)</i> (below): Returns the absolute difference/error, the Root Mean Square Error/Difference and the Maximum difference between two vectors of data X_est, X_true.</li>
</ul>

Get connectivity and measurement information (in functions_base.py):
<ul>
<li> <i>get_connected_elements(net, bus_id_list)</i> (functions_base.py): Returns the id of the elements (lines, trafos and buses) directly connected to buses in bus_id_list.</li>
<li> <i>get_measurements_elmt(net, elmt_id, elmt_type)</i> (functions_base.py): Returns the ids and types of the measurements associated with the element elmt_id.</li>
</ul>
Plotting functions (in functions_plot.py):
<ul>
<li> <i>grid_plot(net)</i>: Shows a plot of the grid with bus ids.</li> 
<li> <i>box_plot(err_list, cases_names, title, ylabel)</i>: Plots the distribution of each list (e.g. error list) in err_list[i]: (e.g. err_list = [[P_err_base, P_err_modified], [Vm_err_base, Vm_err_modified]]).</li>
<li> <i>err_plot(residuals, errors, res_label = "Residuals", err_label = "Measurement Errors", title = "")</i>: Interactive bar plot comparison of vectors of data <i>residuals</i> and <i>meas_errors</i> (i.e. defaulted to compare measurement error and residuals).</li>
</ul>

In [ ]:
def get_estimation(x, Ybus, Yf, Yt, net, slack_bus=6):
    """ Get estimated voltage magnitudes and active power injections from state vector x.
    The state vector x does not include the slack bus angle."""
    # Insert Slack Bus angle
    theta_est = np.insert(x[:len(net.bus)-1], slack_bus, 0)
    Vm_est = x[len(net.bus)-1:]
    V_est = make_V(theta_est, Vm_est)
    S_est = get_PQ_inj(Ybus, V_est); P_est = np.real(S_est); Q_est = np.imag(S_est)
    Sf_est, St_est = get_PQ_flow(net, Yf, Yt, V_est);
    Pf_est = np.real(Sf_est); Qf_est = np.imag(Sf_est)
    Pt_est = np.real(St_est); Qt_est = np.imag(St_est)
    return theta_est, Vm_est, P_est, Q_est, Pf_est, Qf_est, Pt_est, Qt_est

def get_error(X_est, X_true):
    """ Compute RMSE, absolute error and max error for a given estimated and true value vector """
    rmse_X = np.sqrt(np.mean((X_est - X_true)**2))   
    err_X = np.abs(X_est - X_true)
    index_max_err = np.argmax(err_X)
    max_err = err_X[index_max_err]
    return rmse_X, err_X, max_err, index_max_err

In [ ]:
# Functions import
from functions_plot import grid_plot, box_plot, err_plot
from functions_base import get_connected_elements, get_measurements_elmt
id_zones = grid_plot(net)
# Write your code in the code cell(s) below

In [ ]:
### Build your demonstration case here ###
net_copy = copy.deepcopy(net) # Create a save of the original network before modifications